# 02 · pandas essentials

After this notebook you can load, clean, reshape, join and summarise tabular data with pandas 3. You will also be able to turn a raw log of LLM calls into the numbers a production review asks for: p95 latency per model, cost per user and error rate by hour.

**Time:** ~45 min · **Runs:** offline on CPU in < 30 s · **Needs:** [01 · NumPy for AI engineers](01_numpy_for_ai_engineers.ipynb)

## Why this matters in interviews

- AI engineering is full of tables: LLM call logs, eval results, token usage, feedback, datasets for fine-tuning. "Here's a log, find why cost doubled" is a realistic take-home or live exercise.
- `groupby` + named aggregation, `merge` with `validate=`, `pivot_table` and time-series `resample` answer most "summarise this" questions in a few lines.
- pandas 3 changed defaults (copy-on-write, a real string dtype). Knowing why `SettingWithCopyWarning` disappeared, and what replaced it, is a modern gotcha interviewers like.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `dt01` | You need to process a 5 GB CSV on a server with 2 GB of RAM. What do you do? |
| `dt05` | Why convert CSV to Parquet, and when is it worth it? |
| `in18` | How do you calculate cost per request, properly? |
| `pd41` | What is the difference between p50, p95 and p99, and which do you optimise? |
| `po32` | Token usage doubled overnight with no deploy. How do you find the cause? |
| `po23` | What would you put on a monitoring dashboard for a GenAI system? |

In [ ]:
import io
import tempfile
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BLUE, ORANGE, AQUA, GRAY, INK = "#2a78d6", "#eb6834", "#1baf7a", "#b5b3ab", "#52514e"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "axes.axisbelow": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8})
pd.set_option("display.width", 120)
print("pandas", pd.__version__)

## 1. Series and DataFrame

A **Series** is one labelled column: values plus an index. A **DataFrame** is a dict of Series that share one index. Build them from a dict of columns or from a list of row dicts, which is what JSON logs give you.

In [ ]:
latency = pd.Series([820, 1430, 610], index=["req-1", "req-2", "req-3"], name="latency_ms")
print(latency, "\n")

rows = [{"model": "gpt-4.1-mini", "latency_ms": 820, "ok": True},
        {"model": "gpt-4.1", "latency_ms": 1430, "ok": True},
        {"model": "gpt-4.1-mini", "latency_ms": 610, "ok": False}]
df_small = pd.DataFrame(rows)
print(df_small)
print("\ndtypes:", df_small.dtypes.to_dict())
assert df_small["model"].dtype == "str"        # pandas 3: text gets a real string dtype, not `object`

## 2. Reading data: `read_csv`, `head`, `info`, `describe`

`read_csv` accepts a path, a URL or any file-like object. `io.StringIO` lets us keep the data inline. Use `parse_dates=` so timestamps arrive as datetimes rather than strings.

In [ ]:
CSV = """request_id,timestamp,user,model,latency_ms,prompt_tokens,completion_tokens,status,error
r01,2026-09-01 09:02:11,alice,gpt-4.1-mini,812,640,120,ok,
r02,2026-09-01 09:05:40,bob,gpt-4.1,1990,2210,410,ok,
r03,2026-09-01 09:11:02,alice,gpt-4.1-mini,,580,95,timeout,Timeout: no response in 30s
r04,2026-09-01 10:15:37,carol,qwen2.5:3b,2410,900,300,ok,
r05,2026-09-01 10:20:00,bob,gpt-4.1-mini,730,700,150,ok,
r06,2026-09-01 10:48:19,dave,gpt-4.1,2875,3100,520,error,RateLimitError: 429 Too Many Requests
r07,2026-09-01 11:01:55,alice,gpt-4.1-mini,655,610,88,ok,
r08,2026-09-01 11:30:12,carol,qwen2.5:3b,1980,760,210,ok,
r09,2026-09-01 12:04:44,dave,gpt-4.1-mini,901,820,140,ok,
r10,2026-09-01 12:40:03,bob,gpt-4.1,2150,2480,380,ok,
r11,2026-09-01 12:41:03,bob,gpt-4.1,2150,2480,380,ok,
r12,2026-09-01 13:15:29,erin,gpt-4.1-mini,770,,130,error,APIError: 500 Internal Server Error
"""
calls = pd.read_csv(io.StringIO(CSV), parse_dates=["timestamp"])
print(calls.head(3), "\n")
calls.info()

In [ ]:
print(calls.describe().round(1))          # numeric columns: count, mean, std, min, quartiles, max
print("\nshape:", calls.shape, "| timestamp dtype:", calls["timestamp"].dtype, "| user dtype:", calls["user"].dtype)
assert calls.shape == (12, 9) and pd.api.types.is_datetime64_any_dtype(calls["timestamp"])

### Files bigger than memory (`dt01`)

For a 5 GB CSV on a 2 GB machine you **stream it in chunks** and aggregate as you go, reading only the columns you need (`usecols=`) with compact dtypes (`dtype=`). Repeated strings such as model names shrink by 10 to 20× as `category`. For repeated analysis, convert once to Parquet (section 13) and read only the columns you need.

In [ ]:
totals = None
for chunk in pd.read_csv(io.StringIO(CSV), usecols=["model", "prompt_tokens"], dtype={"model": "category"}, chunksize=5):
    part = chunk.groupby("model", observed=True)["prompt_tokens"].sum()
    totals = part if totals is None else totals.add(part, fill_value=0)
print("prompt tokens per model, streamed 5 rows at a time:\n", totals)
in_one_go = calls.groupby("model")["prompt_tokens"].sum()
assert {str(k): v for k, v in totals.items()} == in_one_go.to_dict()      # same answer as loading everything

models = pd.Series(np.random.default_rng(0).choice(["gpt-4.1-mini", "gpt-4.1", "qwen2.5:3b"], 100_000))
mem = {kind: models.astype(kind).memory_usage(deep=True) / 1e6 for kind in ["object", "str", "category"]}
print("\nmemory for 100k model names:", {k: f"{v:.2f} MB" for k, v in mem.items()})
assert mem["category"] < mem["str"] < mem["object"]

## 3. Selecting rows and columns: `loc`, `iloc`, boolean masks

| You want | Write |
|---|---|
| columns by name | `df["col"]`, `df[["a", "b"]]` |
| rows and columns **by label** | `df.loc[row_labels, cols]` (slices *include* the end) |
| rows and columns **by position** | `df.iloc[0:3, 1]` (slices exclude the end, like Python) |
| rows by condition | `df[mask]` or `df.loc[mask, cols]`, combining with `&`, `\|`, `~` and **parentheses** |

In [ ]:
print(calls.loc[calls["status"] != "ok", ["request_id", "model", "status"]], "\n")
print(calls.iloc[:2, [0, 3, 4]], "\n")

slow_gpt41 = calls[(calls["model"] == "gpt-4.1") & (calls["latency_ms"] > 2000)]
print("gpt-4.1 slower than 2 s:", slow_gpt41["request_id"].tolist())
print("users alice or erin    :", calls[calls["user"].isin(["alice", "erin"])].shape[0], "rows")
print("query() reads nicely   :", calls.query("model == 'gpt-4.1-mini' and latency_ms < 800")["request_id"].tolist())
assert slow_gpt41["request_id"].tolist() == ["r06", "r10", "r11"]
assert len(calls.set_index("request_id").loc["r02":"r04"]) == 3        # label slices include the end

## 4. New columns: `assign`, vectorised ops, `.str` and `.dt`

Column arithmetic is vectorised, just like NumPy. `apply(axis=1)` calls a Python function once per row. It is fine for 100 rows and painfully slow for a million. Reach for arithmetic, `np.where`, `.str` and `.dt` first.

In [ ]:
calls = calls.assign(
    total_tokens=calls["prompt_tokens"] + calls["completion_tokens"],
    is_error=calls["status"].ne("ok"),
    hour=calls["timestamp"].dt.hour,
    weekday=calls["timestamp"].dt.day_name(),
    error_type=calls["error"].str.extract(r"^(\w+)", expand=False),     # "RateLimitError: 429 ..." -> "RateLimitError"
    provider=np.where(calls["model"].str.startswith("gpt"), "openai", "local"),
)
print(calls[["request_id", "total_tokens", "is_error", "hour", "error_type", "provider"]].head(6))

big = pd.DataFrame({"p": np.random.default_rng(1).integers(100, 3000, 20_000),
                    "c": np.random.default_rng(2).integers(10, 800, 20_000)})
t0 = time.perf_counter(); slow = big.apply(lambda r: r["p"] * 0.40e-6 + r["c"] * 1.60e-6, axis=1); t_apply = time.perf_counter() - t0
t0 = time.perf_counter(); fast = big["p"] * 0.40e-6 + big["c"] * 1.60e-6; t_vec = time.perf_counter() - t0
print(f"\ncost of 20k rows: apply(axis=1) {t_apply * 1e3:.0f} ms vs vectorised {t_vec * 1e3:.2f} ms ({t_apply / t_vec:.0f}x)")
assert np.allclose(slow, fast) and t_vec < t_apply

## 5. Missing values

`isna()` finds them. `fillna` replaces them and `dropna` removes them. Integer columns with gaps become float (a `NaN` is a float), unless you use the nullable `Int64` dtype. Decide per column: a missing latency on a timeout is *information*, so don't fill it with the mean.

In [ ]:
print("missing per column:\n", calls.isna().sum()[lambda s: s > 0], "\n")
print("prompt_tokens dtype:", calls["prompt_tokens"].dtype, "(a gap forced float)")
print("as nullable Int64  :", calls["prompt_tokens"].astype("Int64").tolist())

answered = calls.dropna(subset=["latency_ms"])                    # latency only exists if a response came back
filled = calls.fillna({"error": "", "prompt_tokens": calls["prompt_tokens"].median()})
print(f"\nrows with a latency: {len(answered)} of {len(calls)}")
assert answered["latency_ms"].notna().all() and filled["error"].notna().all()
assert calls["error"].isna().sum() == 9                            # empty CSV fields in a text column are missing too

## 6. `groupby` with named aggregation

Split, apply, combine. **Named aggregation** (`new_name=(column, function)`) gives you flat, readable column names in one step. `transform` returns a result aligned to the *original* rows. Use it for "each row compared with its group".

In [ ]:
per_model = calls.groupby("model").agg(
    calls=("request_id", "size"),
    error_rate=("is_error", "mean"),
    p50_ms=("latency_ms", "median"),
    p95_ms=("latency_ms", lambda s: s.quantile(0.95)),
    tokens=("total_tokens", "sum"),
).sort_values("p95_ms", ascending=False)
print(per_model.round(2), "\n")
assert per_model["calls"].sum() == len(calls)

calls["latency_vs_model_median"] = calls["latency_ms"] / calls.groupby("model")["latency_ms"].transform("median")
print(calls[["request_id", "model", "latency_ms", "latency_vs_model_median"]].head(5).round(2))

## 7. Reshaping: `pivot_table` and `melt`

`pivot_table` goes from long to wide: one row per index value, one column per category, with an aggregate in each cell. `melt` goes from wide to long, which is the tidy shape that `groupby` and plotting like.

In [ ]:
wide = calls.pivot_table(index="user", columns="model", values="total_tokens", aggfunc="sum", fill_value=0)
print(wide, "\n")
long = wide.reset_index().melt(id_vars="user", var_name="model", value_name="total_tokens")
print(long.head(4))
assert long["total_tokens"].sum() == calls["total_tokens"].sum()

## 8. Combining tables: `merge` (with `validate=`) and `concat`

`merge` is a SQL join. Always pass **`validate=`**. A price table with an accidental duplicate row would silently double those requests' cost, and `validate="many_to_one"` turns that into an error. `indicator=True` shows which rows matched. `concat` stacks tables with the same columns, such as two batches of logs.

In [ ]:
PRICES = pd.DataFrame({"model": ["gpt-4.1-mini", "gpt-4.1", "qwen2.5:3b"],
                       "usd_per_1m_in": [0.40, 2.00, 0.0], "usd_per_1m_out": [1.60, 8.00, 0.0]})   # list prices; local = free
priced = calls.merge(PRICES, on="model", how="left", validate="many_to_one", indicator=True)
priced["cost_usd"] = (priced["prompt_tokens"].fillna(0) * priced["usd_per_1m_in"]
                      + priced["completion_tokens"] * priced["usd_per_1m_out"]) / 1e6
print(priced[["request_id", "model", "cost_usd", "_merge"]].head(4))
assert len(priced) == len(calls) and (priced["_merge"] == "both").all()

bad_prices = pd.concat([PRICES, PRICES.iloc[[0]]], ignore_index=True)       # gpt-4.1-mini listed twice
try:
    calls.merge(bad_prices, on="model", validate="many_to_one")
    raise AssertionError("expected a MergeError")
except pd.errors.MergeError as e:
    print("\nMergeError:", str(e).splitlines()[0])
print("without validate= the join silently grows:", len(calls), "->", len(calls.merge(bad_prices, on="model")), "rows")

inner = calls.merge(PRICES[PRICES["model"] != "qwen2.5:3b"], on="model", how="inner")
print(f"inner join drops unmatched rows: {len(inner)} of {len(calls)} (the qwen rows had no price)")
more_logs = calls.head(2).assign(request_id=["r13", "r14"])
print("concat two batches:", len(pd.concat([calls, more_logs], ignore_index=True)), "rows")

## 9. Sorting, counting and duplicates

In [ ]:
print(calls.sort_values(["model", "latency_ms"], ascending=[True, False])[["model", "latency_ms"]].head(4), "\n")
print(calls["status"].value_counts(), "\n")
print(calls["status"].value_counts(normalize=True).round(3))

content_cols = ["user", "model", "latency_ms", "prompt_tokens", "completion_tokens"]
dupes = calls[calls.duplicated(subset=content_cols, keep=False)]
print("\nsame content, logged twice:", dupes["request_id"].tolist())          # r10 and r11: a client retry logged twice
deduped = calls.drop_duplicates(subset=content_cols, keep="first")
assert len(deduped) == len(calls) - 1

## 10. Recoding and binning: `map`, `replace`, `cut`, `qcut`

`map` translates values through a dict. Values missing from the dict become NaN, so you notice them. `replace` changes only the values you list. `cut` bins on **fixed edges** (your SLO buckets). `qcut` bins on **quantiles**, giving equal-sized groups.

In [ ]:
calls["family"] = calls["model"].map({"gpt-4.1-mini": "small", "gpt-4.1": "large", "qwen2.5:3b": "local"})
calls["status"] = calls["status"].replace({"timeout": "error"})            # treat timeouts as errors
calls["speed"] = pd.cut(calls["latency_ms"], bins=[0, 1000, 2000, np.inf], labels=["fast", "ok", "slow"])
calls["latency_quartile"] = pd.qcut(calls["latency_ms"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
print(calls[["model", "family", "status", "latency_ms", "speed", "latency_quartile"]].head(6))
print("\nspeed buckets:", calls["speed"].value_counts(sort=False).to_dict())
assert calls["family"].notna().all() and (calls["status"] == "timeout").sum() == 0
assert calls["latency_quartile"].value_counts().max() - calls["latency_quartile"].value_counts().min() <= 1

## 11. Time series: `resample` and `rolling`

With a `DatetimeIndex`, `resample("1h")` buckets by clock time, like SQL `GROUP BY hour`, with empty buckets included. `rolling` smooths over a moving window. pandas 3 spells hours as **`"h"`**, and the old `"H"` now raises a `ValueError`.

In [ ]:
by_time = calls.set_index("timestamp").sort_index()
hourly = by_time.resample("1h").agg({"request_id": "count", "total_tokens": "sum", "latency_ms": "median"})
hourly.columns = ["requests", "tokens", "median_latency_ms"]
hourly["tokens_rolling_2h"] = hourly["tokens"].rolling(window=2, min_periods=1).mean()
print(hourly)
assert hourly["requests"].sum() == len(calls)
try:
    by_time.resample("1H")
except ValueError:
    print("\n'1H' is rejected in pandas 3 - use '1h' (and 'min', 's' for minutes and seconds)")

## 12. Nested JSON logs: `json_normalize` and `explode`

LLM gateways and tracing tools log nested JSON: a request, a response with a usage block, and a list of tool calls. `pd.json_normalize` flattens nested dicts into dotted column names. `explode` turns a list column into one row per element. With `record_path=`, `json_normalize` does both at once and returns one row per tool call.

In [ ]:
LOGS = [
    {"id": "t1", "user": {"id": "alice", "plan": "pro"}, "request": {"model": "gpt-4.1-mini", "temperature": 0.2},
     "response": {"usage": {"prompt_tokens": 812, "completion_tokens": 64}, "finish_reason": "tool_calls"},
     "tool_calls": [{"name": "search_docs", "ms": 120}, {"name": "get_order", "ms": 45}]},
    {"id": "t2", "user": {"id": "bob", "plan": "free"}, "request": {"model": "gpt-4.1", "temperature": 0.0},
     "response": {"usage": {"prompt_tokens": 1540, "completion_tokens": 210}, "finish_reason": "stop"},
     "tool_calls": []},
    {"id": "t3", "user": {"id": "alice", "plan": "pro"}, "request": {"model": "gpt-4.1-mini", "temperature": 0.2},
     "response": {"usage": {"prompt_tokens": 950, "completion_tokens": 88}, "finish_reason": "tool_calls"},
     "tool_calls": [{"name": "search_docs", "ms": 98}]},
]
flat = pd.json_normalize(LOGS)
print(flat.columns.tolist(), "\n")
print(flat[["id", "user.id", "request.model", "response.usage.prompt_tokens", "response.finish_reason"]], "\n")

per_tool = flat[["id", "tool_calls"]].explode("tool_calls")
print("explode: one row per list element (an empty list becomes NaN):\n", per_tool, "\n")

tools = pd.json_normalize(LOGS, record_path="tool_calls", meta=["id", ["request", "model"]])
print(tools)
print("\ntool usage:", tools.groupby("name")["ms"].agg(["count", "mean"]).to_dict("index"))
assert len(tools) == 3 and list(tools.columns) == ["name", "ms", "id", "request.model"]

## 13. Saving: CSV vs Parquet (`dt05`)

CSV is text, so types are lost. Timestamps come back as strings, nullable integers as floats, and categories as plain strings. **Parquet** is columnar, compressed and typed. It keeps dtypes, and it lets readers load only the columns they need (`columns=[...]`). That is usually a large saving in size and read time for analytics. pandas uses `pyarrow` for Parquet. The cell falls back gracefully if no engine is installed.

In [ ]:
tmp = Path(tempfile.mkdtemp())
sample = calls[["request_id", "timestamp", "model", "prompt_tokens", "speed"]].assign(
    prompt_tokens=lambda d: d["prompt_tokens"].astype("Int64"))
sample.to_csv(tmp / "calls.csv", index=False)
back_csv = pd.read_csv(tmp / "calls.csv")
print("after CSV round trip    :", {c: str(t) for c, t in back_csv.dtypes.items()})
try:
    sample.to_parquet(tmp / "calls.parquet")
    back_pq = pd.read_parquet(tmp / "calls.parquet")
    print("after Parquet round trip:", {c: str(t) for c, t in back_pq.dtypes.items()})
    assert back_pq.dtypes.equals(sample.dtypes) and back_pq.equals(sample)
    print("read two columns only   :", pd.read_parquet(tmp / "calls.parquet", columns=["model", "prompt_tokens"]).shape)
except ImportError as e:
    print("No Parquet engine here (pip install pyarrow):", e)
assert back_csv["timestamp"].dtype == "str"                 # CSV lost the datetime type

## 14. Copy-on-write in pandas 3: the `SettingWithCopyWarning` gotcha

In pandas 1 and 2, `sub = df[mask]; sub["x"] = 1` *might* modify `df`, depending on whether pandas had made a view or a copy. It raised the infamous `SettingWithCopyWarning`. pandas 3 makes **copy-on-write** the only behaviour. Every indexing result *behaves* like a copy, and data is copied lazily only when you write. So:

- modifying a subset **never** changes the parent;
- chained assignment `df[mask]["col"] = v` **never works**, and pandas warns with `ChainedAssignmentError`;
- to change the parent, use **one** `loc` call: `df.loc[mask, "col"] = v`.

In [ ]:
df = pd.DataFrame({"model": ["gpt-4.1-mini", "gpt-4.1", "qwen2.5:3b"], "flagged": [False, False, False]})

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    df[df["model"] == "gpt-4.1"]["flagged"] = True            # chained assignment: writes to a temporary copy
print("after chained assignment:", df["flagged"].tolist(), "| warning:", [w.category.__name__ for w in caught])
assert not df["flagged"].any() and any(w.category.__name__ == "ChainedAssignmentError" for w in caught)

subset = df[df["model"] != "qwen2.5:3b"]
subset["flagged"] = True                                      # no warning, and the parent is untouched
print("after writing to a subset:", df["flagged"].tolist())
assert not df["flagged"].any()

df.loc[df["model"] == "gpt-4.1", "flagged"] = True           # the correct way
print("after df.loc[mask, col] = :", df["flagged"].tolist())
assert df["flagged"].tolist() == [False, True, False]

## 15. Method chaining

Chain each step on a new line inside parentheses. There are no throwaway variables, and the chain reads top to bottom like the analysis it performs. `assign` with a `lambda` refers to the *current* frame in the chain, and `pipe` slots in your own functions.

In [ ]:
def add_cost(d: pd.DataFrame) -> pd.DataFrame:
    d = d.merge(PRICES, on="model", how="left", validate="many_to_one")
    return d.assign(cost_usd=(d["prompt_tokens"].fillna(0) * d["usd_per_1m_in"] + d["completion_tokens"] * d["usd_per_1m_out"]) / 1e6)

report = (
    calls
    .query("status == 'ok'")
    .pipe(add_cost)
    .groupby("user", as_index=False)
    .agg(requests=("request_id", "size"), tokens=("total_tokens", "sum"), cost_usd=("cost_usd", "sum"))
    .assign(usd_per_1k_requests=lambda d: 1000 * d["cost_usd"] / d["requests"])
    .sort_values("cost_usd", ascending=False)
)
print(report.round(4))

## 16. Capstone: from a raw LLM-usage log to the numbers a review asks for

A generated log of 500 calls over two days. It has three models, eight users, lognormal latencies, and one incident planted in it: on day 1 between 14:00 and 16:00 the hosted provider throws many errors. We want **p95 latency per model**, **cost per user** and **error rate by hour**.

In [ ]:
rng = np.random.default_rng(42)
N = 500
MODEL_MIX = {"gpt-4.1-mini": 0.55, "gpt-4.1": 0.25, "qwen2.5:3b": 0.20}
MEDIAN_MS = {"gpt-4.1-mini": 900, "gpt-4.1": 1700, "qwen2.5:3b": 1300}

log = pd.DataFrame({
    "timestamp": pd.Timestamp("2026-09-01") + pd.to_timedelta(np.sort(rng.uniform(0, 48 * 3600, N)), unit="s"),
    "model": rng.choice(list(MODEL_MIX), N, p=list(MODEL_MIX.values())),
    "user": rng.choice([f"u{i:02d}" for i in range(1, 9)], N),
    "prompt_tokens": rng.lognormal(np.log(700), 0.6, N).round().astype(int),
    "completion_tokens": rng.lognormal(np.log(150), 0.5, N).round().astype(int),
})
log["timestamp"] = log["timestamp"].dt.floor("s")          # whole seconds, like most log timestamps
log["latency_ms"] = (log["model"].map(MEDIAN_MS) * rng.lognormal(0, 0.35, N)
                     + log["completion_tokens"] * 2).round()
incident = log["timestamp"].between("2026-09-01 14:00", "2026-09-01 16:00") & log["model"].str.startswith("gpt")
p_error = np.where(incident, 0.45, 0.02)
log["status"] = np.where(rng.random(N) < p_error, "error", "ok")
day2_u07 = (log["user"] == "u07") & (log["timestamp"] >= "2026-09-02")
log.loc[day2_u07, "prompt_tokens"] *= 6                    # planted for exercise 3: a runaway context on day 2
print(log.head(), "\n")
print(log.shape, "|", log["timestamp"].min(), "->", log["timestamp"].max())

In [ ]:
p95_by_model = log.groupby("model")["latency_ms"].quantile(0.95).sort_values()

cost_by_user = (log.merge(PRICES, on="model", how="left", validate="many_to_one")
                   .assign(cost_usd=lambda d: (d["prompt_tokens"] * d["usd_per_1m_in"]
                                               + d["completion_tokens"] * d["usd_per_1m_out"]) / 1e6)
                   .groupby("user")["cost_usd"].sum().sort_values())

error_rate_hourly = (log.set_index("timestamp")["status"].eq("error")
                        .resample("1h").mean())             # an empty hour gives NaN, not 0

print("p95 latency (ms):", p95_by_model.round(0).to_dict())
print("cost per user ($):", cost_by_user.round(4).to_dict())
worst_hour = error_rate_hourly.idxmax()
print(f"worst hour: {worst_hour} with {error_rate_hourly.max():.0%} errors "
      f"(overall {log['status'].eq('error').mean():.1%})")
assert p95_by_model.idxmax() == "gpt-4.1"
assert pd.Timestamp("2026-09-01 14:00") <= worst_hour < pd.Timestamp("2026-09-01 16:00")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
axes[0].barh(p95_by_model.index, p95_by_model.values, color=BLUE, height=0.55)
axes[0].set_title("p95 latency by model"); axes[0].set_xlabel("ms")

axes[1].barh(cost_by_user.index, cost_by_user.values, color=BLUE, height=0.6)
axes[1].set_title("Cost by user (2 days)"); axes[1].set_xlabel("USD")

axes[2].plot(error_rate_hourly.index, error_rate_hourly.values * 100, color=BLUE, lw=1.5)
axes[2].axvspan(pd.Timestamp("2026-09-01 14:00"), pd.Timestamp("2026-09-01 16:00"), color=ORANGE, alpha=0.15, lw=0)
axes[2].annotate("provider incident", xy=(worst_hour, error_rate_hourly.max() * 100), xytext=(0.45, 0.85),
                 textcoords="axes fraction", fontsize=8, color=INK, arrowprops=dict(arrowstyle="->", color=INK))
axes[2].set_title("Error rate by hour"); axes[2].set_ylabel("% of calls")
axes[2].tick_params(axis="x", labelsize=7, rotation=30)
plt.tight_layout(); plt.show()

Which user tops the cost chart, and why? `u07` is inflated by the day-2 runaway context planted above, but a bar chart can't tell you *when* or *which model*. Exercise 3 finds it the way you would during an incident.

## Try it yourself

**1.** The top 3 users by total tokens (prompt + completion), with their share of all tokens.

In [ ]:
# Solution — try it yourself first, then run this
tokens_by_user = (log.assign(total=lambda d: d["prompt_tokens"] + d["completion_tokens"])
                     .groupby("user")["total"].sum().sort_values(ascending=False))
top3 = tokens_by_user.head(3)
print(pd.DataFrame({"tokens": top3, "share": (top3 / tokens_by_user.sum()).round(3)}))

**2.** p50, p95 and p99 latency per model in one table (hint: `groupby(...).quantile([...]).unstack()`), considering successful calls only.

In [ ]:
# Solution — try it yourself first, then run this
pct = (log[log["status"] == "ok"].groupby("model")["latency_ms"]
          .quantile([0.50, 0.95, 0.99]).unstack().rename(columns=lambda q: f"p{round(q * 100)}"))
print(pct.round(0))
assert (pct["p50"] < pct["p95"]).all() and (pct["p95"] <= pct["p99"]).all()

**3.** "Token usage doubled overnight with no deploy" (`po32`). Compare day 1 with day 2 by user, find who explains the jump, and check which model they used.

In [ ]:
# Solution — try it yourself first, then run this
by_day = (log.assign(day=log["timestamp"].dt.date.astype(str))
             .pivot_table(index="user", columns="day", values="prompt_tokens", aggfunc="sum"))
by_day["change"] = by_day.iloc[:, 1] - by_day.iloc[:, 0]
print(by_day.sort_values("change", ascending=False).head(3), "\n")
culprit = by_day["change"].idxmax()
per_call = log[log["user"] == culprit].groupby(log["timestamp"].dt.strftime("%b %d"))["prompt_tokens"].mean().round(0)
print(f"{culprit}: mean prompt tokens per call by day -> {per_call.to_dict()}")
print("models used on day 2:", log[day2_u07]["model"].value_counts().to_dict())
assert culprit == "u07"

daily = by_day.iloc[:, :2].T                                  # rows = days, columns = users
fig, ax = plt.subplots(figsize=(8, 3.2))
for user in daily.columns:
    is_culprit = user == culprit
    ax.plot(["day 1", "day 2"], daily[user], marker="o", lw=2.5 if is_culprit else 1,
            color=ORANGE if is_culprit else GRAY, zorder=3 if is_culprit else 2)
    ax.text(1.03, daily[user].iloc[1], user, va="center", fontsize=8, color=INK if is_culprit else "#898781")
ax.set_ylabel("prompt tokens per day"); ax.set_xlim(-0.2, 1.25)
ax.set_title(f"One user explains the jump: {culprit}'s prompts grew ~6x overnight")
plt.show()

**4.** Error rate per model per day as a small pivot table, with rates shown as percentages.

In [ ]:
# Solution — try it yourself first, then run this
err = (log.assign(day=log["timestamp"].dt.strftime("%a %d"), is_error=log["status"].eq("error"))
          .pivot_table(index="model", columns="day", values="is_error", aggfunc="mean"))
print((err * 100).round(1).astype(str) + "%")

## Say it in an interview

- **"5 GB CSV, 2 GB RAM?"** (`dt01`) Stream it: `read_csv(chunksize=...)` with `usecols=` and compact `dtype=` (category for repeated strings, smaller ints), and aggregate per chunk. Then convert once to Parquet, so later reads are columnar and typed. If the *aggregate* is too big too, use DuckDB, Polars' streaming engine, or Spark/Dask.
- **"Why Parquet?"** (`dt05`) It is columnar, compressed and typed. You read only the columns you need, dtypes survive the round trip, and files are typically several times smaller than CSV. Keep CSV for small hand-edited files and interchange with non-engineers.
- **"Cost per request, properly?"** (`in18`) Join usage to a price table with `validate="many_to_one"`: prompt × input price + completion × output price. Count retries and failed calls, because you pay for those tokens too. Report per user, per feature and per 1k requests.
- **"p95 per model?"** (`pd41`) `df.groupby("model")["latency_ms"].quantile(0.95)`. Look at the tail, not the mean, and separate successful calls from errors and timeouts.
- **"Token usage doubled, no deploy?"** (`po32`) Slice the change by user, model and day with a pivot and sort by the delta. Then drill into per-call averages. The usual culprits are one tenant, a retry loop, or a context that stopped being trimmed.
- **pandas 3 gotchas:** copy-on-write means writing to a subset never changes the parent, and chained assignment never works (`ChainedAssignmentError`), so use `df.loc[mask, col] = v`. Text columns are `str`, not `object`. Use `"h"` not `"H"` in frequencies. Always `validate=` your merges.
- **Performance:** vectorised column arithmetic beats `apply(axis=1)` by 100× or more. `category` can cut the memory of repeated strings by 10×.

## Next

- [04 · matplotlib & quick charts](04_matplotlib_and_quick_charts.ipynb) turns these aggregates into dashboards, including Power BI-style ones.
- [03 · PyTorch essentials](03_pytorch_essentials.ipynb)
- Put it to work: [latency percentiles p50/p95 & cost](../16_production/05_latency_percentiles_p50_p95_and_cost.ipynb) · [observability, logging & tracing](../16_production/08_observability_logging_and_tracing.ipynb) · [datasets library](../10_huggingface/03_datasets_library.ipynb) · [ML workflow & train/test split](../05_machine_learning/01_ml_workflow_and_train_test_split.ipynb)
- Theory: [AI system design concepts](../../ai_system_design_concepts/index.html) · [interview bank](../../ai_interview_prep/index.html)